In [1]:
import numpy as np
from matchms import Spectrum

spectrum_1 = Spectrum(
    mz=np.array([100, 150, 200.]),
    intensities=np.array([0.7, 0.2, 0.1]),
    metadata={"precursor_mz": 100.0, "test_id": "one"}
)

spectrum_2 = Spectrum(
    mz=np.array([104.9, 140, 190.]),
    intensities=np.array([0.4, 0.2, 0.1]),
    metadata={"precursor_mz": 105.0, "test_id": "two"}
)

spectrum_3 = Spectrum(
    mz=np.array([110, 160, 210.]),
    intensities=np.array([0.5, 0.3, 0.2]),
    metadata={"precursor_mz": 110.0, "test_id": "three"}
)

spectrum_4 = Spectrum(
    mz=np.array([115, 170, 220.]),
    intensities=np.array([0.6, 0.25, 0.15]),
    metadata={"precursor_mz": 115.0, "test_id": "four"}
)

spectrum_5 = Spectrum(
    mz=np.array([110, 160, 210.]),
    intensities=np.array([0.5, 0.3, 0.2]),
    metadata={"precursor_mz": 110.0, "test_id": "five"}
)

spectrum_6 = Spectrum(
    mz=np.array([115, 170, 220.]),
    intensities=np.array([0.6, 0.25, 0.15]),
    metadata={"precursor_mz": 115.0, "test_id": "six"}
)

spectra = [spectrum_1, spectrum_2, spectrum_3, spectrum_4, spectrum_5, spectrum_6]


In [2]:
from matchms import calculate_scores
from matchms.similarity import ModifiedCosine

modified_cosine = ModifiedCosine(tolerance=0.2)
scores = calculate_scores(spectra, spectra, modified_cosine)

print(scores)

StackedSparseArray array of shape (6, 6, 2) containing scores for ('ModifiedCosine_score', 'ModifiedCosine_matches').


In [ ]:
perc=60 #percentage of spectra to keep
n_keep = ((len(spectra)*perc)// 100)  # keep 60% of spectra
print(n_keep)

3


In [5]:
seed = np.random.seed(27)
n_keep = ((len(spectra)*perc)// 100)  # keep 80% of spectra
indices=np.random.choice(len(scores.references), n_keep, replace=False)

In [6]:
indices

array([1, 2, 5])

In [7]:
new_refs = scores.references[indices]
new_queries = scores.queries[indices]

In [8]:
new_refs

array([Spectrum(precursor m/z=105.00, 3 fragments between 104.9 and 190.0),
       Spectrum(precursor m/z=110.00, 3 fragments between 110.0 and 210.0),
       Spectrum(precursor m/z=115.00, 3 fragments between 115.0 and 220.0)],
      dtype=object)

In [9]:
index_map = {old: new for new, old in enumerate(indices)}
print("Index map:", index_map)


Index map: {1: 0, 2: 1, 5: 2}


In [10]:
mask = np.isin(scores._scores.row, indices) & np.isin(scores._scores.col, indices)

old_rows = scores._scores.row[mask]
old_cols = scores._scores.col[mask]
old_data = scores._scores.data[mask]

print("Filtered rows:", old_rows)
print("Filtered cols:", old_cols)
print("Filtered data:", old_data)


Filtered rows: [1 1 1 2 2 2 5 5 5]
Filtered cols: [1 2 5 1 2 5 1 2 5]
Filtered data: [(1.        , 3) (0.70799233, 1) (0.78509387, 1) (0.70799233, 1)
 (1.        , 3) (0.72954057, 1) (0.78509387, 1) (0.72954057, 1)
 (1.        , 3)]


In [11]:
new_rows = np.array([index_map[r] for r in old_rows])
new_cols = np.array([index_map[c] for c in old_cols])

print("New rows:", new_rows)
print("New cols:", new_cols)

New rows: [0 0 0 1 1 1 2 2 2]
New cols: [0 1 2 0 1 2 0 1 2]


In [12]:
from sparsestack import StackedSparseArray

new_stack = StackedSparseArray(len(new_refs), len(new_queries))
#score_name = scores._scores.score_names[0]  # keep the same name
new_stack.add_sparse_data(new_rows, new_cols, old_data, name='')



In [13]:
from matchms import Scores

new_scores = Scores(new_refs, new_queries)
new_scores._scores = new_stack

print("New Scores shape:", new_scores.shape)
print("New references:", new_scores.references)
print("New queries:", new_scores.queries)
print("New sparse rows:", new_scores.scores.row)
print("New sparse cols:", new_scores.scores.col)
print("New sparse data:", new_scores.scores.data)


New Scores shape: (3, 3, 2)
New references: [Spectrum(precursor m/z=105.00, 3 fragments between 104.9 and 190.0)
 Spectrum(precursor m/z=110.00, 3 fragments between 110.0 and 210.0)
 Spectrum(precursor m/z=115.00, 3 fragments between 115.0 and 220.0)]
New queries: [Spectrum(precursor m/z=105.00, 3 fragments between 104.9 and 190.0)
 Spectrum(precursor m/z=110.00, 3 fragments between 110.0 and 210.0)
 Spectrum(precursor m/z=115.00, 3 fragments between 115.0 and 220.0)]
New sparse rows: [0 0 0 1 1 1 2 2 2]
New sparse cols: [0 1 2 0 1 2 0 1 2]
New sparse data: [(1.        , 3) (0.70799233, 1) (0.78509387, 1) (0.70799233, 1)
 (1.        , 3) (0.72954057, 1) (0.78509387, 1) (0.72954057, 1)
 (1.        , 3)]


In [14]:
new_scores

<3x3x2 stacked sparse array containing scores for ('ModifiedCosine_score', 'ModifiedCosine_matches') with 9 stored elements in COOrdinate format>

In [15]:
import sys, os
sys.path.append(os.path.abspath("..")) 

In [16]:
from app.services.SimilarityNetwork import SimilarityNetwork
from app.services.SimilarityNetworkMod import SimilarityNetworkMod


In [17]:
ms_network = SimilarityNetworkMod(
        identifier_key="test_id",
        score_cutoff=0.8,
        max_links=10,
        min_peaks=1,
        link_method='single'  # try 'mutual' too to see difference
    )

In [18]:
ms_network.create_network(new_scores, score_name="ModifiedCosine_score")


In [19]:
print(ms_network.graph)

Graph with 3 nodes and 0 edges


In [20]:
boostrapped_net= ms_network.graph

In [21]:
for u, v, attr in boostrapped_net.edges(data=True):
    print(f"({u}, {v}): {attr}")


In [22]:
print(dict(boostrapped_net.nodes(data=True)))

{'three': {'component': 0}, 'six': {'component': 1}, 'two': {'component': 2}}


In [23]:
dict_bootstrap=dict(boostrapped_net.nodes(data=True))

In [24]:
bootstrap_nodes = list(dict_bootstrap.keys())
print(bootstrap_nodes)

['three', 'six', 'two']


In [25]:
full_network = SimilarityNetworkMod(
        identifier_key="test_id",
        score_cutoff=0.8,
        max_links=10,
        min_peaks=1,
        link_method='single'  # try 'mutual' too to see difference
    )

In [26]:
full_network.create_network(scores, score_name="ModifiedCosine_score")

In [27]:
full_net= full_network.graph

In [28]:
for u, v, attr in full_net.edges(data=True):
    print(f"({u}, {v}): {attr}")

(two, one): {'weight': 0.831479419283098}
(five, one): {'weight': 0.9492481892299481}
(five, three): {'weight': 1.0}
(four, one): {'weight': 0.8567860859069464}
(four, six): {'weight': 1.0}
(three, one): {'weight': 0.9492481892299481}
(six, one): {'weight': 0.8567860859069464}


In [29]:
print(dict(full_net.nodes(data=True)))

{'two': {'component': 0}, 'five': {'component': 0}, 'four': {'component': 0}, 'three': {'component': 0}, 'six': {'component': 0}, 'one': {'component': 0}}


In [30]:
bootstrap_labels = [dict_bootstrap[n]['component'] for n in bootstrap_nodes]
print(bootstrap_labels)

[0, 1, 2]


In [31]:
dict_full=dict(full_net.nodes(data=True))

In [32]:
aligned_full = [dict_full[n]['component'] for n in bootstrap_nodes]
print(aligned_full)

[0, 0, 0]


In [33]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(aligned_full, bootstrap_labels)
nmi = normalized_mutual_info_score(aligned_full, bootstrap_labels)

print("ARI:", ari)
print("NMI:", nmi)

ARI: 0.0
NMI: 0.0


In [34]:
from app.services.LeidenCommunityNetwork import CommunitySimilarityNetwork


community_network_boostrapped = CommunitySimilarityNetwork(
        identifier_key="test_id",
        score_cutoff=0.8,
        k=10,
        remove_intra_community_links=False,
    )


In [35]:
community_network_boostrapped.create_network(new_scores, score_name="ModifiedCosine_score")

KeyError: 'Attribute does not exist'

In [ ]:
community_boots_net = community_network_boostrapped.graph

In [ ]:
dict(community_boots_net.nodes(data=True))

{'two': {'community': 0, 'component': 0},
 'four': {'community': 0, 'component': 0},
 'one': {'community': 0, 'component': 0},
 'three': {'community': 0, 'component': 0}}

In [ ]:
community_network_full= CommunitySimilarityNetwork(
        identifier_key="test_id",
        score_cutoff=0.8,
        k=10,
        remove_intra_community_links=False,
    )

In [ ]:
community_network_full.create_network(scores, score_name="ModifiedCosine_score")

In [ ]:
community_full_net = community_network_full.graph

In [ ]:
dict(community_full_net.nodes(data=True))

{'two': {'community': 0, 'component': 0},
 'four': {'community': 0, 'component': 0},
 'one': {'community': 0, 'component': 0},
 'three': {'community': 0, 'component': 0}}

In [42]:
from sklearn.metrics import adjusted_rand_score

# Suppose you have two clusterings:
full_clusters = [0, 0, 0, 1, 2, 2, 4, 4]  # community labels from full network
bootstrap_clusters = [0, 0, 0, 1, 2, 2, 4, 4]  # from subsampled network

ari = adjusted_rand_score(full_clusters, bootstrap_clusters)
nmi = normalized_mutual_info_score(full_clusters, bootstrap_clusters)
print("ARI:", ari)
print("NMI:", nmi)

ARI: 1.0
NMI: 1.0
